In [ ]:
import os
import numpy as np
import pandas as pd

# ===============================================================
# CONFIG
# ===============================================================
ATTACK_FILE = "./dataset/WADI_attackdataLABLE.csv"

OUT_DIR = "./preprocessed_wadi_supervised/"
os.makedirs(OUT_DIR, exist_ok=True)

LABEL_COL = "Attack LABLE (1:No Attack, -1:Attack)"
DROP_COLS_CANON = {"row", "date", "time"}

# ---- Winsorization (clip quantiles) on RAW values (before MinMax)
USE_WINSOR = True
W_Q_LOW  = 0.001   # 0.1%
W_Q_HIGH = 0.999   # 99.9%

# ===============================================================
# Header finder: skip "0,1,2,3..." fake header
# ===============================================================
def find_header_row(csv_path: str, max_lines: int = 120) -> int:
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
        for i in range(max_lines):
            line = f.readline()
            if not line:
                break
            low = line.lower()
            # robust match
            if ("row" in low) and ("date" in low) and ("time" in low):
                return i
    return 0

def read_wadi_csv(csv_path: str) -> pd.DataFrame:
    header_row = find_header_row(csv_path)
    df = pd.read_csv(
        csv_path,
        header=header_row,
        sep=",",
        engine="c",
        low_memory=False
    )
    df.columns = [str(c).strip() for c in df.columns]
    return df

def drop_non_features(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = []
    for c in df.columns:
        if str(c).strip().lower() in DROP_COLS_CANON:
            drop_cols.append(c)
    return df.drop(columns=drop_cols, errors="ignore")

def to_numeric_and_impute(df_feat: pd.DataFrame) -> pd.DataFrame:
    for c in df_feat.columns:
        s = df_feat[c].astype(str).str.replace(",", ".", regex=False)
        df_feat[c] = pd.to_numeric(s, errors="coerce")

    df_feat = df_feat.dropna(axis=1, how="all")
    df_feat = df_feat.ffill().bfill().fillna(0.0)
    df_feat = df_feat.replace([np.inf, -np.inf], 0.0)
    return df_feat

# ===============================================================
# MinMax scaler "maison"
# ===============================================================
def fit_minmax(X: np.ndarray):
    mn = np.min(X, axis=0)
    mx = np.max(X, axis=0)
    return mn.astype(np.float32), mx.astype(np.float32)

def transform_minmax(X: np.ndarray, mn: np.ndarray, mx: np.ndarray, eps: float = 1e-12):
    denom = (mx - mn)
    denom = np.where(denom < eps, 1.0, denom)
    return ((X - mn) / denom).astype(np.float32)

def pct_out01(X_scaled: np.ndarray) -> float:
    return float(((X_scaled < 0.0) | (X_scaled > 1.0)).mean()) * 100.0

# ===============================================================
# Winsorization helpers (RAW domain) - fit on GLOBAL only
# ===============================================================
def compute_winsor_bounds_raw(X_raw: np.ndarray, q_low: float = 0.001, q_high: float = 0.999):
    low = np.quantile(X_raw, q_low, axis=0)
    high = np.quantile(X_raw, q_high, axis=0)
    low = np.minimum(low, high).astype(np.float32)
    high = np.maximum(low, high).astype(np.float32)
    return low, high

def apply_winsor_raw(X_raw: np.ndarray, low: np.ndarray, high: np.ndarray):
    return np.clip(X_raw, low, high).astype(np.float32)

# ===============================================================
# Loader (Attack file only => GLOBAL)
# ===============================================================
def load_wadi_global(csv_path: str):
    df = read_wadi_csv(csv_path)
    df = drop_non_features(df)

    if LABEL_COL not in df.columns:
        cand = [c for c in df.columns if ("attack" in c.lower()) or ("lable" in c.lower()) or ("label" in c.lower())]
        raise ValueError(
            f"Label column not found. Expected: '{LABEL_COL}'.\n"
            f"Candidates: {cand[:50]}\n"
            f"Columns (first 60): {list(df.columns)[:60]}"
        )

    # label: 1 => No Attack, -1 => Attack
    y_raw = pd.to_numeric(df[LABEL_COL], errors="coerce").to_numpy()
    y_global = (y_raw == -1).astype(np.uint8)

    df = df.drop(columns=[LABEL_COL], errors="ignore")
    df = to_numeric_and_impute(df)

    X_raw = df.to_numpy(dtype=np.float32)
    cols = list(df.columns)
    return X_raw, y_global, cols

# ===============================================================
# MAIN
# ===============================================================
def main():
    Xg_raw, y_global, cols = load_wadi_global(ATTACK_FILE)

    print("GLOBAL raw:", Xg_raw.shape, "| y_global:", y_global.shape,
          "| NaN:", int(np.isnan(Xg_raw).sum()),
          "| Inf:", int(np.isinf(Xg_raw).sum()),
          "| min/max:", float(np.nanmin(Xg_raw)), float(np.nanmax(Xg_raw)))
    print("y_global counts:", {0: int((y_global == 0).sum()), 1: int((y_global == 1).sum())})
    print("F =", len(cols))

    # ---- winsor on RAW (fit on GLOBAL only)
    if USE_WINSOR:
        w_low, w_high = compute_winsor_bounds_raw(Xg_raw, q_low=W_Q_LOW, q_high=W_Q_HIGH)
        Xg_raw_w = apply_winsor_raw(Xg_raw, w_low, w_high)

        print(f"\n✔ Winsor RAW enabled: q_low={W_Q_LOW} q_high={W_Q_HIGH}")
        print("RAW global min/max before:", float(Xg_raw.min()), float(Xg_raw.max()))
        print("RAW global min/max after :", float(Xg_raw_w.min()), float(Xg_raw_w.max()))

        Xg_raw = Xg_raw_w
        np.save(os.path.join(OUT_DIR, "winsor_low_raw.npy"),  w_low)
        np.save(os.path.join(OUT_DIR, "winsor_high_raw.npy"), w_high)

    # ---- minmax on GLOBAL only
    mn, mx = fit_minmax(Xg_raw)
    X_global = transform_minmax(Xg_raw, mn, mx)

    print("\nX_global:", X_global.shape, X_global.dtype,
          "| NaN:", int(np.isnan(X_global).sum()),
          "| Inf:", int(np.isinf(X_global).sum()),
          "| min/max:", float(X_global.min()), float(X_global.max()),
          "| out[0,1]%:", f"{pct_out01(X_global):.4f}%")

    if np.isnan(X_global).any():
        raise RuntimeError("Still NaNs after preprocessing.")

    # ---- save (order preserved)
    np.save(os.path.join(OUT_DIR, "X_global.npy"), X_global)
    np.save(os.path.join(OUT_DIR, "y_global.npy"), y_global.astype(np.uint8))
    np.save(os.path.join(OUT_DIR, "scaler_min.npy"), mn)
    np.save(os.path.join(OUT_DIR, "scaler_max.npy"), mx)
    pd.Series(cols).to_csv(os.path.join(OUT_DIR, "feature_columns.csv"), index=False, header=False)

    print("\n✔ Saved to:", OUT_DIR)

if __name__ == "__main__":
    main()